<a href="https://colab.research.google.com/github/ggalarza1/Real_Estate_data/blob/main/RealEstateMapsAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Google API for Real Estate Intelligence

```markdown
## Interacting with the Google Maps API in Python
To use this code, you will need a **Google Maps API Key** from the Google Cloud Console with the **Distance Matrix API** enabled.
Add your key to Colab Secrets (the key icon on the left) under the name `GOOGLE_MAPS_API_KEY`.
```

In [1]:
!pip install -q googlemaps

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import googlemaps
from datetime import datetime
from google.colab import userdata

# Securely load your Google Maps API key
try:
    API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
    gmaps = googlemaps.Client(key=API_KEY)
except Exception as e:
    print("Please set your GOOGLE_MAPS_API_KEY in the Colab Secrets tab.")
    gmaps = None

if gmaps:
    # Define origins and destinations (can be addresses or coordinates)
    origins = ["Times Square, New York, NY"]
    destinations = ["LaGuardia Airport, Queens, NY"]

    # Define a future departure time (e.g., next Tuesday at 5:30 PM)
    # Note: Traffic prediction requires a departure_time in the future
    departure_time = datetime(2026, 10, 20, 17, 30)

    # Get distance matrix with traffic models
    result = gmaps.distance_matrix(
        origins=origins,
        destinations=destinations,
        mode="driving",
        departure_time=departure_time,
        traffic_model="best_guess"
    )

    # Parse and display travel times
    element = result['rows'][0]['elements'][0]
    if element['status'] == 'OK':
        distance = element['distance']['text']
        duration_typical = element['duration']['text']
        duration_in_traffic = element.get('duration_in_traffic', {}).get('text', 'N/A')

        print(f"Commute from {origins[0]} to {destinations[0]}:")
        print(f"- Distance: {distance}")
        print(f"- Typical Duration: {duration_typical}")
        print(f"- Estimated Duration at 5:30 PM (with traffic): {duration_in_traffic}")
    else:
        print("Error retrieving routing calculations.")

Please set your GOOGLE_MAPS_API_KEY in the Colab Secrets tab.


```markdown
## Interacting with the Google Places API (New!)
Make sure you have enabled the **Places API** in your Google Cloud Console project. The code below demonstrates how to search for local businesses or transit points to help identify popular destinations.
```

In [3]:
if gmaps:
    # Search for popular spots (e.g., transit stations or major hubs) nearby a location
    # We will search near Times Square
    location_coords = (40.7580, -73.9855)

    try:
        # Find transit stations nearby to model where people might commute
        places_result = gmaps.places_nearby(
            location=location_coords,
            radius=1000, # Search radius in meters
            type='subway_station'
        )

        print("Nearby Subway Stations found using Places API:")
        for place in places_result.get('results', [])[:5]:
            name = place.get('name')
            address = place.get('vicinity')
            print(f"- {name} ({address})")

    except Exception as e:
        print(f"An error occurred while calling the Places API: {e}")

```markdown
## Free & Open-Source Alternatives (No API Keys Required)

If you want to build this without configuring billing or API keys, we can use open APIs powered by **OpenStreetMap (OSM)**.

### 1. Estimating Commutes with OSRM (Open Source Routing Machine)
We can request routing and travel durations between coordinates using the public demo server of OSRM.

In [4]:
import requests

# Coordinates for Times Square and LaGuardia Airport (Longitude, Latitude)
start_coords = "-73.9855,40.7580"
end_coords = "-73.8740,40.7769"

# Call the public OSRM API
url = f"http://router.project-osrm.org/route/v1/driving/{start_coords};{end_coords}?overview=false"
response = requests.get(url).json()

if response.get('code') == 'Ok':
    route = response['routes'][0]
    distance_km = route['distance'] / 1000
    duration_mins = route['duration'] / 60

    print(f"Commute Estimation (OSRM):")
    print(f"- Distance: {distance_km:.2f} km")
    print(f"- Estimated Duration: {duration_mins:.1f} minutes (typical traffic)")
else:
    print("Could not retrieve route details.")

Commute Estimation (OSRM):
- Distance: 15.30 km
- Estimated Duration: 20.5 minutes (typical traffic)


```markdown
### 2. Finding Local Places with the Overpass API
We can query OpenStreetMap's Overpass API to find nearby transit stations (like subway stations) within a specific bounding box or radius of coordinates.

In [15]:
import requests

# Utilizing Nominatim with a completely custom User-Agent header
nominatim_url = "https://nominatim.openstreetmap.org/search"

# Broadening the query to 'Times Square, New York' for guaranteed indexing
params = {
    'q': 'Times Square, New York',
    'format': 'json',
    'limit': 5
}

headers = {
    'User-Agent': 'ColabCommuteAppResearchProject_v1_Contact_user@domain.com'
}

try:
    response = requests.get(nominatim_url, params=params, headers=headers, timeout=10)
    if response.status_code == 200:
        data = response.json()
        if data:
            print("Places found near Times Square (OSM Nominatim API):")
            for index, place in enumerate(data, start=1):
                name = place.get('display_name')
                lat = place.get('lat')
                lon = place.get('lon')
                print(f"{index}. {name}")
                print(f"   Coordinates: ({lat}, {lon})\n")
        else:
            print("No results found for your query.")
    else:
        print(f"Nominatim API returned status code: {response.status_code}")
except Exception as e:
    print(f"Could not connect to Nominatim API: {e}")

Places found near Times Square (OSM Nominatim API):
1. Times Square, Manhattan Community Board 5, Manhattan, New York County, New York, 10036, United States
   Coordinates: (40.7570095, -73.9859724)

2. Times Square, Manhattan Community Board 5, Manhattan, New York County, New York, 10036, United States
   Coordinates: (40.7572614, -73.9858998)

3. Times Square, West 42nd Street, Times Square, Manhattan Community Board 5, Manhattan, New York County, New York, 10110, United States
   Coordinates: (40.7555923, -73.9858388)

4. Times Square, Manhattan Community Board 5, Manhattan, New York County, New York, 10036, United States
   Coordinates: (40.7584763, -73.9852945)



```markdown
## Identifying Busy Areas with BestTime.app
To run the code below, sign up for a free developer account at [BestTime.app](https://besttime.app/) and add your API key to Colab Secrets under the name `BESTTIME_API_KEY`.
This script searches for venues in Stamford, CT, and analyzes their predicted busyness level for **Friday at 5 PM (17:00)**.
```

In [21]:
import requests
from google.colab import userdata

# Securely load your BestTime.app API Keys
try:
    # BestTime.app uses 'api_key_private' for backend integration
    API_KEY = userdata.get('BESTTIME_API_KEY')
except Exception:
    print("Please set your BESTTIME_API_KEY in the Colab Secrets tab.")
    API_KEY = None

if API_KEY:
    # BestTime.app multi-venue background search endpoint
    url = "https://besttime.app/api/v1/forecasts/search"

    # BestTime.app API expects parameters in standard form-data format inside the body of a POST request
    payload = {
        'api_key_private': API_KEY,
        'q': 'restaurants, cafes, and bars in Stamford, CT',
        'num': 10
    }

    print("Querying BestTime.app via POST using form payload...")
    response = requests.post(url, data=payload)

    if response.status_code == 200 or response.status_code == 201:
        data = response.json()
        venues = data.get('venues', [])

        print(f"\nVenue Busyness Status in Stamford, CT (Friday at 5:00 PM):\n")
        print(f"{'Venue Name':<35} | {'Address':<35} | {'Busyness %'}")
        print("-" * 85)

        for venue in venues:
            name = venue.get('venue_name', 'Unknown Venue')
            address = venue.get('venue_address', 'Stamford, CT')

            # Friday is index 4 (0=Monday, 6=Sunday), 17:00 is hour index 17 (5 PM)
            forecast = venue.get('forecast', {})

            try:
                # Access Friday's hourly forecast data
                friday_forecast = forecast.get('day_info', {}).get('day_raw', [])[4]
                busy_percentage = friday_forecast[17] # Hour index 17 (5 PM)

                # Formatting labels based on occupancy level
                status = f"{busy_percentage}%"
                if busy_percentage > 75:
                    status += " (Very Busy)"
                elif busy_percentage > 45:
                    status += " (Moderately Busy)"
                else:
                    status += " (Quiet)"

                print(f"{name[:33]:<35} | {address[:33]:<35} | {status}")
            except (IndexError, KeyError, TypeError):
                print(f"{name[:33]:<35} | {address[:33]:<35} | Forecast unavailable")
    else:
        print(f"Error: BestTime.app API returned status code {response.status_code}")
        try:
            print(response.json())
        except Exception:
            print(response.text[:500])
else:
    print("API Key not loaded. Please input your key to run the live analysis.")

Querying BestTime.app via POST using form payload...
Error: BestTime.app API returned status code 405
<!doctype html>
<html lang=en>
<title>405 Method Not Allowed</title>
<h1>Method Not Allowed</h1>
<p>The method is not allowed for the requested URL.</p>

